**오늘의 학습 목표**
- LCEL로 RAG 구축하기
- 간단한 파이프라인 형성하여 흐름 익히기

**학습 날짜**
- 2026.05.18-19

- 단일 LLM : 기본 채팅 모델을 직접 호출
- LCEL : 간단한 체인(ex: prompt + llm + parser)의 경우 사용
- LangGraph : 복잡한 체인(ex: 분기, 사이클, 여러 에이전트)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
# RecursiveCharacterTextSplitter
"""
긴 문서를 한번에 Vector DB에 넣지 않고,
일정한 chunk_size 기준으로 잘라(chunks 생성)
embedding 후 저장하기 위한 Text Splitter
"""
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
# Qdrant 
"""
로컬 환경에서 구축 가능한 Vector DB
문서 chunk의 embedding vector를 저장하고,
query embedding과의 유사도 기반 검색을 수행한다.
"""
from langchain_qdrant import qdrant
from langchain_core.output_parsers import StrOutputParser # 답변을 string을 받음
from langchain_core.runnables import RunnablePassthrough

##### 01. PyPDF로 문서에서 텍스트 추출

In [2]:
loader = PyPDFLoader(
    file_path="data/PIKC.pdf", # 내 논문을 예시로..
)
docs = loader.load() # 페이지 기준으로 자르기
len(docs)

11

In [3]:
docs[0] # document 형식 확인

Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-09-01T21:22:59+05:30', 'moddate': '2023-09-13T14:37:35-04:00', 'ieee article id': '10179190', 'trapped': 'False', 'ieee issue id': '10251409', 'subject': 'IEEE Sensors Journal;2023;23;18;10.1109/JSEN.2023.3292288', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '7361', 'title': 'Classification Network-Guided Weighted <italic>K</italic>-Means Clustering for Multitouch Detection', 'source': 'data/PIKC.pdf', 'total_pages': 11, 'page': 0, 'page_label': '21397'}, page_content='IEEE SENSORS JOURNAL, VOL. 23, NO. 18, 15 SEPTEMBER 2023 21397\nClassification Network-Guided Weighted\nK-Means Clustering for Multitouch Detection\nJames Lee, Jun-Ha Yun, Jae-Hun Shim\n , and Suk-Ju Kang\n ,Member, IEEE\nAbstract —In mobile devices, multito

In [4]:
# 200자씩 끊어서 가져와보기
for doc in docs:
    print(doc.page_content[:200])
    print(doc.metadata)
    print("-"*100)

IEEE SENSORS JOURNAL, VOL. 23, NO. 18, 15 SEPTEMBER 2023 21397
Classification Network-Guided Weighted
K-Means Clustering for Multitouch Detection
James Lee, Jun-Ha Yun, Jae-Hun Shim
 , and Suk-Ju Kang
{'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-09-01T21:22:59+05:30', 'moddate': '2023-09-13T14:37:35-04:00', 'ieee article id': '10179190', 'trapped': 'False', 'ieee issue id': '10251409', 'subject': 'IEEE Sensors Journal;2023;23;18;10.1109/JSEN.2023.3292288', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '7361', 'title': 'Classification Network-Guided Weighted <italic>K</italic>-Means Clustering for Multitouch Detection', 'source': 'data/PIKC.pdf', 'total_pages': 11, 'page': 0, 'page_label': '21397'}
------------------------------------------------------------------------------------------

In [6]:
# 1000자씩 자르는데, 맥락 유지를 위해 200자는 겹쳐서 자르기
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs) # document 객체 splitter

# splits[0] => 첫 번째 document 객체
# page_content => 실제 텍스트 내용
print(splits[0].page_content)

IEEE SENSORS JOURNAL, VOL. 23, NO. 18, 15 SEPTEMBER 2023 21397
Classification Network-Guided Weighted
K-Means Clustering for Multitouch Detection
James Lee, Jun-Ha Yun, Jae-Hun Shim
 , and Suk-Ju Kang
 ,Member, IEEE
Abstract —In mobile devices, multitouch recognition tech-
nology is utilized to interact with the touch screen. Recogniz-
ing the coordinates of finger touches is the most important
task in multitouch recognition. Several studies have been
conducted to detect touch coordinates. However, these stud-
ies have been unable to detect exact coordinates when
the big-finger problem occurs. The big-finger problem is a
phenomenon in which a hole is generated in the center of
the touch area when a large object approaches the touch
screen, and it becomes more serious in a low-ground-mass
environment. To solve the big-finger problem, we propose a
novel method that detects touch coordinates using the clus-
tering algorithm. We perform clustering on the touch data by


In [10]:
len(splits)

72

##### 02. Ollama Embedding으로 bge-m3 기반 텍스트 임베딩
- 텍스트 → 의미를 담은 벡터(vector)로 변환

In [11]:
embeddings = OllamaEmbeddings(model="bge-m3")

##### 03. Qdrant 벡터DB(인메모리)에 임베딩 저장

In [12]:
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4


# 로컬 디스크에 Qdrant DB 생성 => 서버 없이 로컬 RAG 가능
client = QdrantClient(path="/tmp/PIKC")

# embedding_PIKC 라는 벡터 저장 공간 생성
# 처음만 돌리고 추후 주석처리
client.create_collection(
     collection_name="embedding_PIKC",
     # 1024 : 벡터DB 사이즈 (embedding 모델과 맞아야 함)
     vectors_config=VectorParams(size=1024, distance=Distance.COSINE), 
 )

qdrant = QdrantVectorStore(
    client=client,
    collection_name="embedding_PIKC",
    embedding=embeddings,
    # retrieval mode가 SPARSE인 경우 어려운 용어나 특정 도메인이 정확할때 사용
    retrieval_mode=RetrievalMode.DENSE,
)

# 각 chunk(Document 객체)마다 고유 ID 부여
# 처음만 돌리고 추후 주석처리
uuids = [str(uuid4()) for _ in range(len(splits))]

qdrant.add_documents(documents=splits, ids=uuids)


['870f3c14-5803-44b5-abef-10b13d5e40bd',
 '1f23dec0-b0fc-419f-8f50-1ef33aa08603',
 '7f2502fb-2c8f-4df6-a105-6ae61e285355',
 '356ebed6-43b5-4599-ac56-70d4eb2640c6',
 '37628404-7f56-4c2e-a32e-1367e54974a5',
 'ed4b9b27-b530-411e-891c-202208b67fed',
 '9c265aae-a51a-42d3-8343-b1e6040ac9ee',
 '2f01745b-3a5a-4320-81ac-3e5813297c23',
 'bb0e4c26-cb49-426f-8726-477774e7a959',
 '6735cb38-b9f3-4e96-a9bb-8d45f3d5394c',
 '3da2684c-ce6f-4207-af4c-c1ee33e8ded0',
 '5ee27d82-3282-4513-b133-7fa2887f07a1',
 '3548b6c2-fa92-4616-88d8-92143d58ff56',
 'c223ad68-8e7e-45bc-ac3d-3816f729e4a6',
 'a031139a-8df6-4eeb-aef0-1ba501c1f51c',
 '89665b10-be1f-479a-baaf-dcbaa985d431',
 'eb0a9be4-ad17-4693-ab3e-43ba2a9d5545',
 'f98939cb-98bb-4f35-9c66-37e402f6e590',
 '07c8dc77-b533-40dd-bdbc-f04530ca51b0',
 'd2cd72d9-dd37-4307-b426-90e5c81089c3',
 'ca8e237f-2897-4e1a-bbc0-fbee0b88881b',
 'c8eb4a2a-07c4-43ed-8329-e1fddbed1ec8',
 '01e09723-1e62-40ee-b2e6-6916bcc0f4d3',
 'be933e82-a57a-44c7-9937-9efa44d6c6aa',
 '919a41db-25b4-

##### 04. 벡터DB 기반 Retriever 구현
- 내 질문과 가장 관련 있는 chunk들을 Vector DB에서 찾아 출력

In [14]:
#retriever = qdrant.as_retriever() #as_retriever => 검색기

#search_result=retriever.invoke("Big Finger Problem이 뭐야?", k=5)
retriever = qdrant.as_retriever(
    search_kwargs={"k": 2}
)

search_result = retriever.invoke(
    "Big Finger Problem이 뭐야?"
)

for doc in search_result:
    print(doc.page_content[:500])
    print(doc.metadata)
    print("-"*100)

21398 IEEE SENSORS JOURNAL, VOL. 23, NO. 18, 15 SEPTEMBER 2023
Fig. 1. Touch signal with a large area of touch causes the big-finger
problem that generates holes at the center of the touch.
the touch coordinates with the topmost pixel in the generated
touch mask.
Recent studies have utilized deep-learning technology for
performance improvement in touch coordinate detection [17],
[18], [19]. Yoon et al. [17] used a convolutional neural network
(CNN) to classify the number of touches generated on 
{'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-09-01T21:22:59+05:30', 'moddate': '2023-09-13T14:37:35-04:00', 'ieee article id': '10179190', 'trapped': 'False', 'ieee issue id': '10251409', 'subject': 'IEEE Sensors Journal;2023;23;18;10.1109/JSEN.2023.3292288', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publica

##### 05. RAG 프롬프트 템플릿 설정

In [15]:
# langchain.prompts에서 langchain_core.prompts로 변경
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 주어진 질문에 답변하는 똑똑한 학자입니다.

컨텍스트:
{context}

질문:
{question}

답변:
""")

##### 06. 단순 답변 Streaming하기

{"변수명": 값, "변수명": 값}
| prompt

In [16]:
from IPython.display import Markdown
from langchain_core.runnables import RunnableParallel

llm = ChatOllama(model="gemma3:4b", temperature=0)


def format_docs(docs): # 검색한 결과 하나의 string으로 엮어줌
    return "\n\n".join(doc.page_content for doc in docs)

"""
전체 구조 : "context","question' | prompt | llm | StrOutputParser()

1) "context" : retriever | format_docs
- retriever은 04의 벡터 DB 기반 retriever를 의미
- format docs는 바로 위에 검색 결과를 string으로 묶어주는 것

2) "question"
- RunnablePassthrough()는 아무 처리 안 하고 그대로 전달하는 것 의미
- Big Finger Problem이 뭐야? 를 그대로 전달

3) prompt는 05에서 설정한 템플릿

4) llm은 위 코드에서 ollama 기반 model="gemma3:4b"

5) StrOutputParser() : 문자열 추출
"""
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

for chunk in rag_chain.stream("Big Finger Problem이 뭐야?"):
    print(chunk, end='', flush=True)

Big Finger Problem은 터치스크린에서 여러 개의 손가락이 동시에 터치될 때 발생하는 문제입니다. 이로 인해 터치스크린의 중앙에 구멍(hole)이 생겨 정확한 터치 좌표를 파악하기 어려워지는 현상입니다. 즉, 여러 손가락이 동시에 터치될 때, 각 손가락의 터치 영역이 겹쳐져 터치 좌표 감지 오류를 유발하는 것입니다.

오 정확한데..

##### 부록 : 참고한 소스를 함께 스트리밍하기


In [ ]:
"""
입력 질문 (Big Finger Problem이 뭐야?)
   ↓
 ┌──────────────┐
 │              │
retriever    passthrough
 │              │
context      question
"""

In [17]:
# 답변만 받으면 신뢰성이 낮다
# 따라서 참고한 소스를 스트리밍하여 참고한다

rag_chain_from_docs = (

        # RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
        # - 처음 context는 Document 리스트인데, prompt에는 문자열이 들어가야함
        # - [Document1, Document2, Document3] => "Document1 내용\n\nDocument2 내용\n\nDocument3 내용" 으로 변환

    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | prompt
    | llm
    | StrOutputParser()
)

# RunnableParallel로 검색 결과와 질문을 병렬로 처리 => 답변을 생성하는 최종 체인
rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)
"""
* assign(answer=rag_chain_from_docs)
 {"context": retriever, "question": RunnablePassthrough()} 에
 아래를 answer로 추가한다는 뜻
rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | prompt
    | llm
    | StrOutputParser()
)
"""

for chunk in rag_chain_with_source.stream("Big Finger Problem이 뭐야?"):
    if "context" in chunk:
        print("Retrieved Documents:")
        for i, doc in enumerate(chunk["context"]):
            print(f"소스 [{i+1}]: <{doc.metadata['title']}>")
            print(f"참고 내용: {doc.page_content[:100]}...\n")
        print("-" * 80)
    elif "answer" in chunk:
        print(chunk["answer"], end="", flush=True)

Retrieved Documents:
소스 [1]: <Classification Network-Guided Weighted <italic>K</italic>-Means Clustering for Multitouch Detection>
참고 내용: 21398 IEEE SENSORS JOURNAL, VOL. 23, NO. 18, 15 SEPTEMBER 2023
Fig. 1. Touch signal with a large are...

소스 [2]: <Classification Network-Guided Weighted <italic>K</italic>-Means Clustering for Multitouch Detection>
참고 내용: leads to inaccurate touch coordinate detection.
To resolve these challenges, we propose a novel mult...

--------------------------------------------------------------------------------
Big Finger Problem은 터치스크린에서 여러 개의 손가락이 동시에 터치될 때 발생하는 문제입니다. 이로 인해 터치스크린의 중앙에 구멍(hole)이 생겨 정확한 터치 좌표를 파악하기 어려워지는 현상입니다. 즉, 여러 손가락이 동시에 터치될 때, 각 손가락의 터치 영역이 겹쳐져 터치 좌표 감지 오류를 유발하는 것입니다.

공부 끗!